In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from IPython.display import Audio
import IPython.display as ipd
from scipy.io import wavfile
import tempfile
import os
import librosa
import pandas as pd
import seaborn as sns
import h5py
import mne
from scipy.stats import zscore
from mne_bids import BIDSPath, read_raw_bids
from matplotlib_venn import venn2,venn2_circles
from tqdm import tqdm

In [2]:
cm = 1/2.54
plt.rcParams['svg.fonttype'] = 'none'

fontdict = dict(fontsize=7)
fontsize = 7

red = '#A9373B'
blue = '#2369BD'
orange = '#CC8963'
green = '#009944'

stg_color = '#20B2AA'
smc_color = '#6A5ACD'
insula_color = '#D4AF37'

reds = sns.light_palette(red, as_cmap=True)
blues = sns.light_palette(blue, as_cmap=True)
oranges = sns.light_palette(orange, as_cmap=True)
greens = sns.light_palette(green, as_cmap=True)

recon_dir = '/cwork/ns458/ECoG_Recon/'
mne.viz.set_3d_backend('notebook')                    # MNE 3D in-notebook static backend
# text svg

Using notebook 3d backend.


# Load significant HGA

In [3]:
import pandas as pd
import numpy as np
from mne_bids import BIDSPath
from tqdm import tqdm

task = [
        'PhonemeSequence', 
        'LexicalDelay',
        'PictureNaming',
        'SentenceRep',
        ]

ref = 'bipolar'
hga_paths = []
for t in task:
    hga_paths.extend(
        BIDSPath(
            root=f'../results/{t}({ref})',
            datatype='HGA',
            suffix='time',  # 注意这里改成 'time'
            check=False,
        ).match()
    )

# 2. 筛选 perception 和 passive 的数据
HGAs = []
for path in tqdm(hga_paths):
    df = pd.read_csv(path)
    # 只保留 perception 或 passive
    HGAs.append(df)

HGAs = pd.concat(HGAs)

# rename both phase : Resp and Response to Resp
HGAs.loc[HGAs.phase == 'Resp', 'phase'] = 'Response'
HGAs.loc[HGAs.phase == 'Audio', 'phase'] = 'Stimulus'

# phase all convert to lower
HGAs['phase'] = HGAs['phase'].str.lower()

# rename INS to 'Insula'
HGAs.loc[HGAs.roi == 'INS', 'roi'] = 'Insula'
# combine HG to STG
# HGAs.loc[HGAs.roi == 'HG', 'roi'] = 'STG'
# rename CG to Cingulate cortex
HGAs.loc[HGAs.roi == 'PrG', 'roi'] = 'SMC'
HGAs.loc[HGAs.roi == 'PoG', 'roi'] = 'SMC'
HGAs.loc[HGAs.roi == 'Subcentral', 'roi'] = 'SMC'
HGAs.loc[HGAs.roi == 'HG', 'roi'] = 'STG'


HGAs.loc[HGAs.label.isin([
    'ctx_lh_G_and_S_cingul-Mid-Post',
    'ctx_rh_G_and_S_cingul-Mid-Post',
    'ctx_lh_G_cingul-Post-dorsal'
    ]), 'roi'] = 'PCC'


dacc_labels = [
    'ctx_lh_G_and_S_cingul-Mid-Ant',
    'ctx_rh_G_and_S_cingul-Mid-Ant'
]
HGAs.loc[HGAs.label.isin(dacc_labels), 'roi'] = 'dACC' # 这里的 dACC 就是最严格的 SN 枢纽
# rename CG to ACC
HGAs.loc[HGAs.roi=='CG', 'roi'] = 'ACC'

HGAs.head()

100%|██████████| 2077/2077 [00:16<00:00, 129.46it/s]


,time,channel,value,mask,roi,hemi,subject,description,task,phase,modality,label,x,y,z
0,-1.0,D0019_ROG13-14,-0.043502,False,mOccG,R,D0019,Repeat,PhonemeSequence,stimulus,sound,ctx_rh_G_occipital_middle,41.472074,-88.150203,-3.545099
1,-1.0,D0019_RIIH5-6,-0.054793,False,OPC,R,D0019,Repeat,PhonemeSequence,stimulus,sound,ctx_rh_Pole_occipital,11.886836,-94.046488,10.403848
2,-1.0,D0019_ROG4-5,-0.025007,False,sOccG,R,D0019,Repeat,PhonemeSequence,stimulus,sound,ctx_rh_G_occipital_sup,16.298644,-91.860079,29.446276
3,-1.0,D0019_ROG18-19,-0.083139,False,Intersection,R,D0019,Repeat,PhonemeSequence,stimulus,sound,Intersection,41.531682,-78.171196,19.605156
4,-1.0,D0019_RLSO7-8,-0.040828,False,PhG,R,D0019,Repeat,PhonemeSequence,stimulus,sound,ctx_rh_G_oc-temp_med-Parahip,18.457267,-10.274114,-31.949923


In [4]:
# load a exlude pandas df
fpath = '../results/exlude_insula.csv'
exclude_df = pd.read_csv(fpath, index_col=0)
# exclude NaN columns and flatten into single list
exclude_chn = [electrode for item in exclude_df.Exclude.dropna() 
                for electrode in eval(item)]

HGAs = HGAs[~HGAs.channel.isin(exclude_chn)]
print(exclude_chn)

['D0032_LAI4-5', 'D0040_L1IF3-4', 'D0084_RFAI1-2', 'D0086_LTPI2-3', 'D0090_RIA4-5', 'D0096_LFAI2-3', 'D0096_LFAI4-5', 'D0102_RFAI2-3', 'D0106_LTAS2-3', 'D0121_LFMI3-4', 'D0122_LFAI3-4', 'D0125_LIA4-5', 'D0125_LIA7-8']


In [5]:
# Insula region classification function for row-wise application
def classify_insula_row(row, y_threshold=0):
    """
    Classify a single insula electrode into AIC or PIC
    Designed for use with df.apply()
    
    Returns: 'AIC', 'PIC', or original roi if not insula
    """
    if row['roi'] != 'Insula':
        return row['roi']
    
    label = row['label']
    y_coord = row['y']
    
    # AIC classification conditions
    if ('G_insular_short' in label or 
        'S_circular_insula_ant' in label or
        ('S_circular_insula_sup' in label and y_coord > y_threshold) or
        ('S_circular_insula_inf' in label and y_coord > y_threshold)):
        return 'AIC'
    
    # PIC classification conditions  
    elif ('G_Ins_lg_and_S_cent_ins' in label or
          ('S_circular_insula_sup' in label and y_coord <= y_threshold) or
          ('S_circular_insula_inf' in label and y_coord <= y_threshold)):
        return 'PIC'
    
    # If no conditions match, return original
    return row['roi']

# Apply classification to entire DataFrame (function handles filtering internally)
HGAs['roi'] = HGAs.apply(classify_insula_row, axis=1, y_threshold=0)

# Statistics results
print(f"AIC electrode count: {len(HGAs[HGAs['roi'] == 'AIC'])}")
print(f"PIC electrode count: {len(HGAs[HGAs['roi'] == 'PIC'])}")

# Check unclassified electrodes
unclassified = HGAs[HGAs['roi'] == 'Insula']
if len(unclassified) > 0:
    print(f"Unclassified electrode count: {len(unclassified)}")
    print("Unclassified label types:", unclassified['label'].unique())
    
HGAs.head()

AIC electrode count: 444800
PIC electrode count: 399104


,time,channel,value,mask,roi,hemi,subject,description,task,phase,modality,label,x,y,z
0,-1.0,D0019_ROG13-14,-0.043502,False,mOccG,R,D0019,Repeat,PhonemeSequence,stimulus,sound,ctx_rh_G_occipital_middle,41.472074,-88.150203,-3.545099
1,-1.0,D0019_RIIH5-6,-0.054793,False,OPC,R,D0019,Repeat,PhonemeSequence,stimulus,sound,ctx_rh_Pole_occipital,11.886836,-94.046488,10.403848
2,-1.0,D0019_ROG4-5,-0.025007,False,sOccG,R,D0019,Repeat,PhonemeSequence,stimulus,sound,ctx_rh_G_occipital_sup,16.298644,-91.860079,29.446276
3,-1.0,D0019_ROG18-19,-0.083139,False,Intersection,R,D0019,Repeat,PhonemeSequence,stimulus,sound,Intersection,41.531682,-78.171196,19.605156
4,-1.0,D0019_RLSO7-8,-0.040828,False,PhG,R,D0019,Repeat,PhonemeSequence,stimulus,sound,ctx_rh_G_oc-temp_med-Parahip,18.457267,-10.274114,-31.949923


# Video Generation Configuration

In [6]:
import pyvista as pv
import matplotlib.colors as mcolors
from mne.viz import Brain
from scipy.spatial import cKDTree
import imageio

# Configuration parameters
WINDOW_SIZE = 0.1  # seconds
STEP_SIZE = 0.02   # seconds
TIME_START = -0.5  # seconds
TIME_END = 1.2     # seconds
FPS = 20
BRAIN_WIDTH = 800  # pixels
BRAIN_HEIGHT = 400 # pixels
TIMELINE_HEIGHT = 10  # pixels

# Output path
OUTPUT_PATH = '../img/delay_activity.mp4'

print(f"Configuration:")
print(f"  Time range: {TIME_START} to {TIME_END} seconds")
print(f"  Window size: {WINDOW_SIZE} seconds")
print(f"  Step size: {STEP_SIZE} seconds")
print(f"  Expected frames: {int((TIME_END - TIME_START - WINDOW_SIZE) / STEP_SIZE) + 1}")
print(f"  Output: {OUTPUT_PATH}")

Configuration:
  Time range: -0.5 to 1.2 seconds
  Window size: 0.1 seconds
  Step size: 0.02 seconds
  Expected frames: 80
  Output: ../img/delay_activity.mp4


# Helper Functions

In [7]:
def get_time_window_data(HGAs, t_start, t_end):
    """
    Aggregate HGA data for a specific time window.
    
    Parameters:
    -----------
    HGAs : pd.DataFrame
        Full HGA dataframe
    t_start : float
        Window start time in seconds
    t_end : float
        Window end time in seconds
    
    Returns:
    --------
    pd.DataFrame
        Spatial dataframe with mean HGA per channel
    """
    # Filter data for this time window
    window_mask = (
        (HGAs['phase'] == 'delay') &
        (HGAs['description'] == 'Repeat') &
        (HGAs['time'] >= t_start) &
        (HGAs['time'] <= t_end) &
        (HGAs['mask'] == True)
    )
    
    window_data = HGAs[window_mask]
    
    # Aggregate by channel (mean HGA value)
    spatial_data = (
        window_data.groupby('channel')
        .agg({
            'value': 'mean',
            'x': 'first',
            'y': 'first',
            'z': 'first',
            'roi': 'first',
            'label': 'first',
            'hemi': 'first',
            'subject': 'first',
        })
        .reset_index()
        .rename(columns={'value': 'HGA'})
    )
    
    # Exclude unwanted ROIs
    exclude_rois = [
        'Intersection', 'OPC', 'Cun', 'CollatAnt',
        'PhG', 'FuG', 'LinGs', 'Hipp', 'Central', 'mOccG',
        'LinG', 'PCuns', 'PCun', 'GRect', 'FuGs', 'Amyg', 'Calcarine',
        'Put', 'OccS', 'VDC', 'LatV', 'InfLatV', 'iOccGs',
        'Intersection(unknown)', 'Thal', 'BrainStem(unknown)',
        'Paracentral', 'PCC', 'Sylvian', 'OFCs'
    ]
    spatial_data = spatial_data[~spatial_data.roi.isin(exclude_rois)]
    
    return spatial_data

In [8]:
def render_brain_frame(spatial_data, vmin, vmax, t_start, t_end, time_range):
    """
    Render a single frame showing bilateral brain views with timeline.
    
    Parameters:
    -----------
    spatial_data : pd.DataFrame
    Spatial data with HGA values per channel
    vmin, vmax : float
    Global normalization range for HGA values
    t_start, t_end : float
    Current time window
    time_range : tuple
    (min_time, max_time) for full timeline
    
    Returns:
    --------
    np.ndarray
    RGB image array
    """
    if len(spatial_data) == 0:
        # Return empty frame if no data
        total_height = BRAIN_HEIGHT + TIMELINE_HEIGHT
        return np.ones((total_height, BRAIN_WIDTH, 3), dtype=np.uint8) * 255
    
    coords = spatial_data[['x', 'y', 'z']].values
    hga_values = spatial_data['HGA'].values
    
    # Normalization and mapping functions (match fig2 style but smaller dots)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    alpha_min, alpha_max = 0.1, 1
    size_min, size_max = 2.5, 12.0
    
    def hga_to_size(hga):
        normalized = np.clip((hga - vmin) / (vmax - vmin), 0, 1)
        return size_min + normalized * (size_max - size_min)
    
    def hga_to_alpha(hga):
        normalized = np.clip((hga - vmin) / (vmax - vmin), 0, 1)
        return alpha_min + normalized * (alpha_max - alpha_min)
    
    # Load fsaverage surfaces and KD-trees
    lh_pial_coords, _ = mne.read_surface(f"{recon_dir}/fsaverage/surf/lh.pial")
    rh_pial_coords, _ = mne.read_surface(f"{recon_dir}/fsaverage/surf/rh.pial")
    lh_tree = cKDTree(lh_pial_coords)
    rh_tree = cKDTree(rh_pial_coords)
    
    mask_lh = coords[:, 0] < 0
    mask_rh = coords[:, 0] > 0
    
    # Helper to add electrodes
    def add_electrodes(brain, coords, hga_vals, tree, pial_coords):
        _, indices = tree.query(coords)
        coords_proj = pial_coords[indices]
        for pt, hga in zip(coords_proj, hga_vals):
            rgb = reds(norm(hga))[:3]
            alpha = hga_to_alpha(hga)
            size = hga_to_size(hga)
            cloud = pv.PolyData(pt[None, :])
            brain._renderer.plotter.add_mesh(
                cloud,
                render_points_as_spheres=True,
                point_size=size,
                color=rgb,
                opacity=alpha,
                lighting=False,
            )
    
    # Create Brain objects
    lh_brain = Brain(
        "fsaverage", subjects_dir=recon_dir, surf="pial",
        hemi="lh", background="white", show=False,
        cortex=(0.9, 0.9, 0.9), alpha=0.3, size=(BRAIN_WIDTH//2, BRAIN_HEIGHT)
    )
    rh_brain = Brain(
        "fsaverage", subjects_dir=recon_dir, surf="pial",
        hemi="rh", background="white", show=False,
        cortex=(0.9, 0.9, 0.9), alpha=0.3, size=(BRAIN_WIDTH//2, BRAIN_HEIGHT)
    )
    
    # Add electrodes
    if mask_lh.any():
        add_electrodes(lh_brain, coords[mask_lh], hga_values[mask_lh], lh_tree, lh_pial_coords)
    if mask_rh.any():
        add_electrodes(rh_brain, coords[mask_rh], hga_values[mask_rh], rh_tree, rh_pial_coords)
    
    # Set view
    lh_brain.show_view(azimuth=180, elevation=90, distance=350)
    rh_brain.show_view(azimuth=0, elevation=90, distance=350)
    
    # Get screenshots
    lh_img = lh_brain.screenshot(mode="rgb")
    rh_img = rh_brain.screenshot(mode="rgb")
    
    # Close brains
    lh_brain.close()
    rh_brain.close()
    
    # Create combined figure with timeline
    fig = plt.figure(figsize=(BRAIN_WIDTH/100, (BRAIN_HEIGHT + TIMELINE_HEIGHT)/100), dpi=300)
    
    # Brain views
    ax_lh = plt.subplot2grid((10, 2), (0, 0), rowspan=8)
    ax_rh = plt.subplot2grid((10, 2), (0, 1), rowspan=8)
    
    ax_lh.imshow(lh_img)
    ax_lh.axis('off')
    
    ax_rh.imshow(rh_img)
    ax_rh.axis('off')
    
    # Timeline
    ax_timeline = plt.subplot2grid((10, 2), (8, 0), colspan=2, rowspan=2)
    
    # Draw timeline
    ax_timeline.set_xlim(time_range[0], time_range[1])
    ax_timeline.set_ylim(0, 1)
    
    # Timeline axis
    ax_timeline.axhline(0.5, color='black', linewidth=1)
    
    # Tick marks
    tick_times = np.arange(time_range[0], time_range[1] + 0.5, 0.5)
    for t in tick_times:
        ax_timeline.plot([t, t], [0.4, 0.6], 'k-', linewidth=1)
        ax_timeline.text(t, 0.2, f'{t:.1f}', ha='center', va='top', fontsize=8)
    
    # Highlight current window
    ax_timeline.axvspan(t_start, t_end, alpha=0.3, color=red, zorder=2)
    
    # Window center marker
    t_center = (t_start + t_end) / 2
    ax_timeline.plot([t_center, t_center], [0.3, 0.7], color=red, linewidth=2, zorder=3)
    
    ax_timeline.set_yticks([])
    ax_timeline.set_xticks([])
    ax_timeline.spines['top'].set_visible(False)
    ax_timeline.spines['right'].set_visible(False)
    ax_timeline.spines['left'].set_visible(False)
    ax_timeline.set_xlabel('Time (s)', fontsize=9)
    
    plt.tight_layout()
    
    # Convert to numpy array (robust to backend differences)
    fig.canvas.draw()
    # Use renderer buffer_rgba to avoid missing tostring_rgb in some backends
    buf = np.asarray(fig.canvas.renderer.buffer_rgba())
    frame = buf[:, :, :3].copy()  # drop alpha
    
    plt.close(fig)
    
    return frame

# Compute Global Normalization Range

In [9]:
# Calculate time windows
TIME_WINDOWS = []
t = TIME_START
while t <= TIME_END - WINDOW_SIZE:
    TIME_WINDOWS.append((t, t + WINDOW_SIZE))
    t += STEP_SIZE

print(f"Total time windows: {len(TIME_WINDOWS)}")

# Delay-window quantile normalization to down-weight big stimulus peaks
DELAY_NORM_WINDOW = (0.6, 1)
NORM_QUANTILES = (0.05, 0.95)

# Collect HGA values for delay window only (Repeat, delay, masked)
delay_mask = (
    (HGAs['phase'] == 'delay') &
    (HGAs['description'] == 'Repeat') &
    (HGAs['time'] >= DELAY_NORM_WINDOW[0]) &
    (HGAs['time'] <= DELAY_NORM_WINDOW[1]) &
    (HGAs['mask'] == True)
)

# raw HGA values column is 'value'
delay_vals = HGAs.loc[delay_mask, 'value'].to_numpy()
delay_vals = delay_vals[~np.isnan(delay_vals)]

# Collect all HGA values across all windows as fallback
all_hga_values = []
print("Computing global normalization range...")
for t_start, t_end in tqdm(TIME_WINDOWS, desc="Sampling windows"):
    spatial_data = get_time_window_data(HGAs, t_start, t_end)
    if len(spatial_data) > 0:
        all_hga_values.extend(spatial_data['HGA'].values)

all_hga_values = np.array(all_hga_values)
all_hga_values = all_hga_values[~np.isnan(all_hga_values)]

if delay_vals.size > 0:
    vmin, vmax = np.quantile(delay_vals, NORM_QUANTILES)
    source_label = f"delay window {DELAY_NORM_WINDOW} q{int(NORM_QUANTILES[0]*100)}–q{int(NORM_QUANTILES[1]*100)}"
else:
    # fallback to overall 2–98% if delay window empty
    vmin, vmax = np.quantile(all_hga_values, (2, 98))
    source_label = "global q2–q98 (delay window empty)"

print(f"Normalization range from {source_label}: vmin={vmin:.3f}, vmax={vmax:.3f}")
print(f"Total data points used: {len(delay_vals) if delay_vals.size > 0 else len(all_hga_values)}")

Total time windows: 80
Computing global normalization range...


Sampling windows: 100%|██████████| 80/80 [01:04<00:00,  1.24it/s]

Normalization range from delay window (0.6, 1) q5–q95: vmin=-0.294, vmax=0.786
Total data points used: 48791


# Generate Video Frames

In [10]:
frames = []
time_range = (TIME_START, TIME_END)

print(f"Generating {len(TIME_WINDOWS)} frames...")

for t_start, t_end in tqdm(TIME_WINDOWS, desc="Rendering frames"):
    # Get data for this window
    spatial_data = get_time_window_data(HGAs, t_start, t_end)
    
    # Render frame
    frame = render_brain_frame(spatial_data, vmin, vmax, t_start, t_end, time_range)
    frames.append(frame)

print(f"Generated {len(frames)} frames")
if frames:
    print(f"Frame shape: {frames[0].shape}")

Generating 80 frames...


Rendering frames: 100%|██████████| 80/80 [05:05<00:00,  3.82s/it]

Generated 80 frames
Frame shape: (1230, 2400, 3)


# Save Video

In [11]:
print(f"Saving video to {OUTPUT_PATH}...")

# Save as MP4 with imageio using ffmpeg writer
with imageio.get_writer(
    OUTPUT_PATH,
    format='FFMPEG',
    mode='I',
    fps=FPS,
    codec='libx264'
) as writer:
    for frame in frames:
        # ensure uint8 contiguous
        writer.append_data(np.ascontiguousarray(frame, dtype=np.uint8))

print(f"Video saved successfully!")
print(f"Duration: {len(frames) / FPS:.2f} seconds")
print(f"Resolution: {frames[0].shape[1]}x{frames[0].shape[0]}")
print(f"FPS: {FPS}")

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2400, 1230) to (2400, 1232) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Saving video to ../img/delay_activity.mp4...
Video saved successfully!
Duration: 4.00 seconds
Resolution: 2400x1230
FPS: 20
